<a href="https://colab.research.google.com/github/SeungYunIm/notebooks/blob/main/IBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. JSONL 파일 읽어오기 (한 줄씩 읽어야 하므로 lines=True 필수)
df = pd.read_json('/content/drive/MyDrive/iso_sensors_mcqa_val.jsonl', lines=True)

# 2. 데이터가 어떻게 생겼는지 상위 5개만 확인
df.head()

,id,question_type,passage,question,options,answer,metadata
0,qw7Vajtf68mJ,open_ended_multi_choice,"Consider the passage and question, then indica...","For electric motor, which failure event is not...","{'A': 'Loss of input power phase', 'B': 'Brush...",A,"{'asset_class': 'electric motor', 'family': 'n..."
1,qylWbqKj7hMq,open_ended_multi_choice,"Read the given passage, question and select th...",The speed reading of a aero gas turbine is abn...,"{'A': 'Compressor damaged', 'B': 'Power turbin...",C,"{'asset_class': 'aero gas turbine', 'family': ..."
2,q4mIDJZwwqgO,open_ended_multi_choice,"Study the passage, then answer the question by...","For power transformer, if on-load tap-changer ...","{'A': 'Torque', 'B': 'Leak reactance fluX', 'C...",G,"{'asset_class': 'power transformer', 'family':..."
3,qs8ybj9UtX3E,open_ended_multi_choice,"Read the given passage, question and select th...","Out of the failure events below, which one lea...",{'A': 'Overheating/ auxiliary cooling system f...,F,"{'asset_class': 'power transformer', 'family':..."
4,q1MogBmvuJzc,open_ended_multi_choice,"Consider the passage and question, then indica...",A condition-monitoring engineer suspects low o...,"{'A': 'Resistance', 'B': 'Leak reactance fluX'...",D,"{'asset_class': 'power transformer', 'family':..."


In [ ]:
# metadata에서 asset_class만 쏙 뽑아서 카운트하기
df['asset_class'] = df['metadata'].apply(lambda x: x.get('asset_class'))
# df 데이터프레임에 asset_class라는 이름의 새로운 컬럼을 하나 만들겠다는 선언으로 기존의 metadata 열은 단순 글자가 아닌 딕셔너리 형태로 되어 있음
# apply를 통해 df의 metadata 행 하나하나에 대해 똑같이 훑을 수 있음
# lamda 함수 정의를 통해 metadata 상자 한 개에 대해 asset_class 라벨값만 꺼내오라는 선언

print(df['asset_class'].value_counts())

asset_class
power transformer                           380
reciprocating internal combustion engine    120
compressor                                  110
aero gas turbine                            108
industrial gas turbine                      104
electric motor                               98
electric generator                           94
steam turbine                                82
fan                                          80
pump                                         66
Name: count, dtype: int64


In [ ]:
df['asset_class'].head()

,asset_class
0,electric motor
1,aero gas turbine
2,power transformer
3,power transformer
4,power transformer


In [ ]:
# 질문에 'not' 또는 'NOT'이 포함된 행만 필터링
negation_questions = df[df['question'].str.contains('not|NOT', case=False)]
# 세로대 기호 |는 OR을 의미
# case=False를 통해 대소문자 구별을 중지

print(f"전체 문제 중 부정형 문제 개수: {len(negation_questions)}개")

# 부정형 문제 예시 3개만 출력해보기
negation_questions['question'].head(3)

In [ ]:
# 객관식 보기의 개수 확인
df['n_options'] = df['metadata'].apply(lambda x: x.get('n_options'))
print(df['n_options'].value_counts())

In [ ]:
# 기계 종류(asset_class)별 정답(answer) 개수 크로스탭 표 만들기
pivot_df = pd.crosstab(df['asset_class'], df['answer'])
# crosstab이란 두 가지 항목을 서로 교차시켜서 개수를 세어주는 표 형성 명령어
# 각각 세로축과 가로축 항목이 됨

print("=== 기계 종류별 정답 분포 ===")
print(pivot_df)

In [ ]:
# metadata에서 family만 추출하여 카운트
df['problem_family'] = df['metadata'].apply(lambda x: x.get('family'))

print("=== 데이터셋 문제 유형 비율 ===")
print(df['problem_family'].value_counts())

In [ ]:
# 글자 수 컬럼 생성
df['passage_len'] = df['passage'].str.len()
df['question_len'] = df['question'].str.len()

print("=== 텍스트 길이 통계 ===")
print(f"지문(Passage) 평균 글자 수: {df['passage_len'].mean():.1f} 자")

# 가장 긴 지문의 글자 수도 확인
print(f"지문(Passage) 최대 글자 수: {df['passage_len'].max()} 자")
print(f"질문(Question) 평균 글자 수: {df['question_len'].mean():.1f} 자")

In [ ]:
import pandas as pd

# 1. 원본 JSONL 파일 읽어오기
df = pd.read_json('/content/iso_sensors_mcqa_val.jsonl', lines=True)

# 2. 엑셀 파일로 변환하여 저장하기
df.to_excel('대회_데이터_변환본.xlsx', index=False)

In [ ]:
import pandas as pd

# 1. 데이터 읽어오기
df = pd.read_json('/content/iso_sensors_mcqa_val.jsonl', lines=True)

# 2. metadata 딕셔너리를 각각의 독립된 열로 쪼개기
metadata_columns = df['metadata'].apply(pd.Series)

# 3. 쪼갠 열들을 기존 표에 합치고, 원래 있던 metadata 열은 지우기
df = pd.concat([df, metadata_columns], axis=1).drop(columns=['metadata'])

# 4. 엑셀 파일로 깨끗하게 내보내기
df.to_excel('대회_데이터_메타데이터분리.xlsx', index=False)